# Dataset

In [5]:
text = ""
with open("tinyshakesphere.txt") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
vocab = list(set(text))
vocab_size = len(vocab)
print(vocab_size)

65


In [7]:
encode = {val:i for i,val in enumerate(vocab)}
decode = {i:val for i,val in enumerate(vocab)}

print(decode[0])

t


In [8]:
import torch
tokenized_text = torch.tensor([encode[c] for c in text],dtype=torch.long)
print(tokenized_text[:1000])

tensor([17, 39, 26, 16,  0, 64,  4, 39,  0, 39,  6, 61,  9, 42, 49, 24, 61, 46,
        44, 26, 61, 64, 29, 61, 64, 45, 26, 44, 35, 61, 61, 63, 64, 33,  9,  3,
        64, 46,  2, 26,  0, 37, 61, 26, 47, 64, 37, 61, 33, 26, 64, 23, 61, 64,
        16, 45, 61, 33,  5, 28, 49, 49, 34, 10, 10, 42, 49, 55, 45, 61, 33,  5,
        47, 64, 16, 45, 61, 33,  5, 28, 49, 49, 17, 39, 26, 16,  0, 64,  4, 39,
         0, 39,  6, 61,  9, 42, 49, 60, 44,  2, 64, 33, 26, 61, 64, 33, 10, 10,
        64, 26, 61, 16, 44, 10, 59, 61, 63, 64, 26, 33,  0, 37, 61, 26, 64,  0,
        44, 64, 63, 39, 61, 64,  0, 37, 33,  9, 64,  0, 44, 64, 46, 33, 23, 39,
        16, 37, 13, 49, 49, 34, 10, 10, 42, 49, 52, 61, 16, 44, 10, 59, 61, 63,
        28, 64, 26, 61, 16, 44, 10, 59, 61, 63, 28, 49, 49, 17, 39, 26, 16,  0,
        64,  4, 39,  0, 39,  6, 61,  9, 42, 49, 17, 39, 26, 16,  0, 47, 64,  3,
        44,  2, 64,  5,  9, 44, 29, 64,  4, 33, 39,  2, 16, 64, 62, 33, 26, 35,
        39,  2, 16, 64, 39, 16, 64, 35, 

In [9]:
train_data = tokenized_text[:int(0.9*len(text))]
val_data = tokenized_text[int(0.9*len(text)):]

print(len(train_data),len(val_data))

1003854 111540


In [10]:
block_size = 128

def get_batch(split):
    data = train_data if split == 'train' else val_data

    random_starting_indices = torch.randint(len(data) - block_size, (block_size,))

    x = torch.stack([data[i:i+block_size] for i in random_starting_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_starting_indices])
    return x,y
    

x,y = get_batch("train")
print(x)
print(y)


tensor([[16, 64, 63,  ..., 47, 64, 29],
        [ 9,  5, 64,  ...,  2,  0, 64],
        [ 3, 64, 16,  ...,  9, 44,  0],
        ...,
        [16, 45, 61,  ..., 51,  0, 64],
        [39, 10, 10,  ..., 64, 33, 64],
        [49, 60, 44,  ..., 61, 64, 46]])
tensor([[64, 63, 44,  ..., 64, 29, 39],
        [ 5, 64, 23,  ...,  0, 64, 37],
        [64, 16, 29,  ..., 44,  0, 64],
        ...,
        [45, 61, 35,  ...,  0, 64,  9],
        [10, 10, 64,  ..., 33, 64, 16],
        [60, 44,  2,  ..., 64, 46, 39]])


# Bigram Model

In [11]:
import torch.nn as nn
import torch.nn.functional as F

class BigramModel(nn.Module):
    def __init__(self,n_embeddings, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=n_embeddings, embedding_dim=vocab_size)

    def forward(self,input_encodings,targets_encodings=None):
        input_embeddings = self.embedding(input_encodings) # (B,T,C) , B = block_size, T = length of sequence, C = embedding_dim

        loss = None
        if targets_encodings is not None:
            input_embeddings_for_loss = input_embeddings.permute(0,2,1) # (B,C,T)
            loss = F.cross_entropy(input_embeddings_for_loss,targets_encodings)
        
        return loss, input_embeddings 
    
    def generate(self,input_encoding, max_output_len):
        for _ in range(max_output_len):
            _,logits = self(input_encoding)
            logits = logits[:,-1,:] # Last char of sequence
            predictions_probs = F.softmax(logits,dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding,prediction),dim=1)

        return input_encoding
    
model = BigramModel(vocab_size, vocab_size) # 1 row for each possible char, and for softmax, we need the vocab as output

start_context = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))



todBwYo,PYWpMyNkKutMygqHRaakwFn-aIOiDoUTrInGmh'HebylqXnAXQkhKZEXHmTt-eP!TmD
WtMABRxRtY&$FGyHttHiy;a&S


# Training Bigram

In [12]:
device = "cuda" if torch.cuda.is_available else 'cpu'
print(device)

model = model.to(device)

cuda


In [13]:
def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 100 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")


import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
train(model, 4000, optimizer,device)

Step 0 | Train Loss = 4.7179
Step 100 | Train Loss = 4.5690
Step 200 | Train Loss = 4.4255
Step 300 | Train Loss = 4.3004
Step 400 | Train Loss = 4.1615
Step 500 | Train Loss = 4.0359
Step 600 | Train Loss = 3.9387
Step 700 | Train Loss = 3.8321
Step 800 | Train Loss = 3.7388
Step 900 | Train Loss = 3.6309
Step 1000 | Train Loss = 3.5402
Step 1100 | Train Loss = 3.4603
Step 1200 | Train Loss = 3.3793
Step 1300 | Train Loss = 3.3116
Step 1400 | Train Loss = 3.2433
Step 1500 | Train Loss = 3.1670
Step 1600 | Train Loss = 3.1186
Step 1700 | Train Loss = 3.0753
Step 1800 | Train Loss = 3.0291
Step 1900 | Train Loss = 2.9675
Step 2000 | Train Loss = 2.9148
Step 2100 | Train Loss = 2.8902
Step 2200 | Train Loss = 2.8568
Step 2300 | Train Loss = 2.8070
Step 2400 | Train Loss = 2.7953
Step 2500 | Train Loss = 2.7490
Step 2600 | Train Loss = 2.7364
Step 2700 | Train Loss = 2.7181
Step 2800 | Train Loss = 2.6769
Step 2900 | Train Loss = 2.6736
Step 3000 | Train Loss = 2.6512
Step 3100 | Train Lo

In [14]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(model,device)
start_context = torch.zeros((1, 1), dtype=torch.long).to(device)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


Validation Loss =  2.556445837020874
t t,
Wifo s's be QUS:
Wheas by;
DWh,
t NGble
IO:

WeinAnovI sut lqliou$k.:
Whomen t o&zpAs woreg ade-


# Transformer Blackbox Model

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlackbox(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            batch_first=True 
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.lin = nn.Linear(d_model, vocab_size)

    def forward(self, input_encodings, target_encodings=None):
        B, T = input_encodings.shape

        
        tok_emb = self.token_embedding(input_encodings) # (B, T) -> (B, T, d_model)
        
        positions = torch.arange(0, T, dtype=torch.long, device=input_encodings.device)
        pos_emb = self.pos_embedding(positions) 

        x = tok_emb + pos_emb 
        
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_encodings.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        
        logits = self.lin(x) # (B, T, vocab_size)
        loss = None

        if target_encodings is not None:
            logits_for_loss = logits.permute(0, 2, 1)
            loss = F.cross_entropy(logits_for_loss, target_encodings)

        return loss, logits

    def generate(self, input_encoding, max_output_len):
        for _ in range(max_output_len):
            input_cond = input_encoding[:, -self.block_size:] 
            
            _, logits = self(input_cond)
            logits = logits[:, -1, :] 
            predictions_probs = F.softmax(logits, dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding, prediction), dim=1)

        return input_encoding

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
blackbox_model = TransformerBlackbox(vocab_size, 512, 8, 8, block_size)
blackbox_model = blackbox_model.to(device)

In [19]:
import torch.optim as optim

def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 5 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")

optimizer = optim.AdamW(blackbox_model.parameters(), lr=1e-4, weight_decay=0.01)
train(blackbox_model, 400, optimizer,device)

Step 0 | Train Loss = 2.3909
Step 5 | Train Loss = 2.5056
Step 10 | Train Loss = 2.4396
Step 15 | Train Loss = 2.4208
Step 20 | Train Loss = 2.4018
Step 25 | Train Loss = 2.3956
Step 30 | Train Loss = 2.4112
Step 35 | Train Loss = 2.3639
Step 40 | Train Loss = 2.3742
Step 45 | Train Loss = 2.3536
Step 50 | Train Loss = 2.3491
Step 55 | Train Loss = 2.3207
Step 60 | Train Loss = 2.3148
Step 65 | Train Loss = 2.3127
Step 70 | Train Loss = 2.2948
Step 75 | Train Loss = 2.2982
Step 80 | Train Loss = 2.2896
Step 85 | Train Loss = 2.2875
Step 90 | Train Loss = 2.2576
Step 95 | Train Loss = 2.2651
Step 100 | Train Loss = 2.2374
Step 105 | Train Loss = 2.2531
Step 110 | Train Loss = 2.2238
Step 115 | Train Loss = 2.2186
Step 120 | Train Loss = 2.2266
Step 125 | Train Loss = 2.1968
Step 130 | Train Loss = 2.1747
Step 135 | Train Loss = 2.1751
Step 140 | Train Loss = 2.1994
Step 145 | Train Loss = 2.1740
Step 150 | Train Loss = 2.1599
Step 155 | Train Loss = 2.1312
Step 160 | Train Loss = 2.1476

In [20]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(blackbox_model,device)

inp_text = "Hello"
inp_text_tokenized = torch.tensor([encode[c] for c in inp_text], dtype=torch.long).unsqueeze(0).to(device)
gen = blackbox_model.generate(inp_text_tokenized, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


Validation Loss =  1.899012804031372
Hellookesslice
Ofied, ploose ime? a bearmal aBut his tur?


QUEEN:
ENaw you you, thou shim; trost heart,



In [22]:

torch.save(blackbox_model.state_dict(), 'blackbox_transformer_model.pt')
print("Model successfully saved!")

Model successfully saved!
